In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    
    for file in files:
        
        if file.endswith(".csv"):
            print(os.path.join(root, file))

In [ ]:
# loading dataset
df = pd.read_csv("/kaggle/input/datasets/nalisha/tesla-ea-deliveries-and-production-data20152025/tesla_deliveries_dataset_2015_2025.csv")

In [ ]:
# first 5 rows

df.head()

In [ ]:
# last 5 rows

df.tail()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
# Missing Values Check
missing_values = df.isnull().sum()

print(missing_values)

In [ ]:
# Duplicate Rows Check
duplicate_rows = df.duplicated().sum()

print("Duplicate Rows :", duplicate_rows)

In [ ]:
# Data Quality Report
report = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.values,
    "Missing Values": df.isnull().sum().values,
    "Unique Values": df.nunique().values
})

report

In [ ]:
df.describe()

In [ ]:
# Separate Numerical and Categorical Features
num_cols = []
cat_cols = []

for col in df.columns:

    if df[col].dtype == "object":
        cat_cols.append(col)
    else:
        num_cols.append(col)

print("Numerical Features")
print(num_cols)

print()

print("Categorical Features")
print(cat_cols)

# EDA Phase 1

In [ ]:
# Numerical Features
num_cols = [
    "Year",
    "Month",
    "Production_Units",
    "Avg_Price_USD",
    "Battery_Capacity_kWh",
    "Range_km",
    "CO2_Saved_tons",
    "Charging_Stations"
]

In [ ]:
# Categorical Features
cat_cols = [
    "Region",
    "Model",
    "Source_Type"
]

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    df["Estimated_Deliveries"],
    kde=True
)

plt.title("Estimated Deliveries Distribution")

plt.show()

In [ ]:
print(df["Estimated_Deliveries"].describe())

In [ ]:
# Numerical Features Distribution
for col in num_cols:

    plt.figure(figsize=(6,4))

    sns.histplot(
        df[col],
        kde=True
    )

    plt.title(col)

    plt.show()

In [ ]:
# Outlier Detection
for col in num_cols:

    plt.figure(figsize=(6,2))

    sns.boxplot(
        x=df[col]
    )

    plt.title(col)

    plt.show()

In [ ]:
# Correlation Analysis
corr_matrix = df.corr(numeric_only=True)

plt.figure(figsize=(10,8))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm"
)

plt.show()

In [ ]:
# Deliveries Correlation
corr_matrix["Estimated_Deliveries"]\
.sort_values(
    ascending=False
)

In [ ]:
# Categorical Analysis
# Deliveries by Region
plt.figure(figsize=(10,5))

sns.barplot(
    data=df,
    x="Region",
    y="Estimated_Deliveries"
)

plt.xticks(rotation=45)

plt.show()

In [ ]:
# Deliveries by Model
plt.figure(figsize=(10,5))

sns.barplot(
    data=df,
    x="Model",
    y="Estimated_Deliveries"
)

plt.xticks(rotation=45)

plt.show()

In [ ]:
# Time Trend
monthly_sales = (
    df.groupby(["Year"])
    ["Estimated_Deliveries"]
    .mean()
)

plt.figure(figsize=(10,5))

monthly_sales.plot(
    marker="o"
)

plt.ylabel("Average Deliveries")

plt.show()

In [ ]:
corr_matrix["Estimated_Deliveries"]\
.sort_values(
    ascending=False
)

# Feature Engineering

In [ ]:
df_ml = df.copy()

In [ ]:
 # Feature 1: Year-Month
df_ml["Year_Month"] = (
    df_ml["Year"].astype(str)
    + "-"
    + df_ml["Month"].astype(str)
)

In [ ]:
# quarter feature

df_ml["Quarter"] = ((df_ml["Month"] - 1) // 3) + 1

In [ ]:
# month cyclic encoding

import numpy as np

df_ml["Month_sin"] = np.sin(
    2 * np.pi * df_ml["Month"] / 12
)

df_ml["Month_cos"] = np.cos(
    2 * np.pi * df_ml["Month"] / 12
)

In [ ]:
# price per km

df_ml["Price_Per_KM"] = (
    df_ml["Avg_Price_USD"] /
    df_ml["Range_km"]
)

In [ ]:
# co2 saved per charging station

df_ml["CO2_Per_Station"] = (
    df_ml["CO2_Saved_tons"] /
    df_ml["Charging_Stations"]
)

In [ ]:
df_ml.head()

In [ ]:
df_ml.columns.tolist()

In [ ]:
df_ml.drop(
    columns=["Production_Efficiency"],
    inplace=True
)

In [ ]:
df_ml.columns.tolist()

# Encoding

In [ ]:
cat_cols = [
    "Region",
    "Model",
    "Source_Type"
]

In [ ]:
 # One Hot Encoding
df_encoded = pd.get_dummies(
    df_ml,
    columns=cat_cols,
    drop_first=True
)

In [ ]:
print(df_encoded.shape)

In [ ]:
df_encoded.columns.tolist()

In [ ]:
# Target aur Features
X = df_encoded.drop("Estimated_Deliveries", axis=1)

y = df_encoded["Estimated_Deliveries"]

In [ ]:
print(X.select_dtypes(include="object").columns.tolist())

In [ ]:
# remove string column

X = X.drop("Year_Month", axis=1)

In [ ]:
# Train Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

# Evaluation

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(y_test, y_pred)

print("MAE :", round(mae, 2))
print("RMSE :", round(rmse, 2))
print("R2 Score :", round(r2, 4))

In [ ]:
# Random Forest
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

In [ ]:
# Evaluation
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import numpy as np

print("MAE :", round(mean_absolute_error(y_test, rf_pred), 2))

print("RMSE :", round(
    np.sqrt(mean_squared_error(y_test, rf_pred)), 2
))

print("R2 :", round(
    r2_score(y_test, rf_pred), 4
))

# Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {
    "n_estimators": [50, 100, 200],
    "max_depth": [5, 10, 15]
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    params,
    cv=3,
    scoring="r2"
)

grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)

# Time Series Forecasting

In [ ]:
monthly_data = (
    df.groupby(["Year", "Month"])
    ["Estimated_Deliveries"]
    .mean()
    .reset_index()
)

monthly_data.head()

In [ ]:
# Create Data Column
monthly_data["Date"] = pd.to_datetime(
    monthly_data["Year"].astype(str)
    + "-"
    + monthly_data["Month"].astype(str)
    + "-01"
)

In [ ]:
monthly_data = monthly_data.sort_values("Date")

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    monthly_data["Date"],
    monthly_data["Estimated_Deliveries"]
)

plt.title("Tesla Deliveries Trend")

plt.show()

# ==========================================================
# Tesla Sales & Deliveries Forecasting - ML Project
# ==========================================================

# Step 1: Loaded the Tesla deliveries dataset from Kaggle
# and explored its structure, columns, and data types.

# Step 2: Performed data preprocessing by checking
# missing values, duplicate records, and overall
# data quality to ensure the dataset was clean.

# Step 3: Conducted Exploratory Data Analysis (EDA)
# to understand delivery patterns, feature distributions,
# and relationships between variables.

# Step 4: Analyzed the target variable
# (Estimated_Deliveries) and performed correlation
# analysis to identify important features.

# Step 5: Created new features to improve model performance:
# - Year_Month
# - Quarter
# - Month_sin
# - Month_cos
# - Price_Per_KM
# - CO2_Per_Station

# Step 6: Converted categorical variables into numerical
# format using One-Hot Encoding.

# Step 7: Selected Estimated_Deliveries as the target
# variable and separated features (X) and target (y).

# Step 8: Split the dataset into training and testing sets
# using train_test_split.

# Step 9: Built a Linear Regression model as a baseline
# model for delivery prediction.

# Step 10: Evaluated model performance using:
# - Mean Absolute Error (MAE)
# - Root Mean Squared Error (RMSE)
# - R2 Score

# Linear Regression Results:
# MAE  : 309.07
# RMSE : 383.48
# R2   : 0.9901

# Step 11: Built and compared additional regression models
# such as Random Forest Regressor.

# Step 12: Applied Hyperparameter Tuning using GridSearchCV
# to find the best model parameters.

# Step 13: Performed Time Series Analysis using Year and Month
# information to study delivery trends over time.

# Step 14: Visualized historical delivery patterns and
# analyzed future sales forecasting trends.

# Final Outcome:
# Successfully developed an end-to-end Machine Learning
# pipeline covering data preprocessing, EDA, feature
# engineering, regression modeling, hyperparameter tuning,
# model evaluation, and time series forecasting for Tesla
# sales and delivery prediction.